# Coffee Standard J25 — AF2LUM-SAFE seed 42

Single final-package screen dari official YOLO26n: shared luminance AF2, stochastic cue strength, semantic-safe augmentation, dan source-identity repeat sampling. Output dan `last.pt` disimpan ke Drive. Test tidak diekstrak atau dibuka.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import csv, importlib, json, os, shutil, subprocess, sys, time
from pathlib import Path
ARM='AF2LUMSAFE'; BRANCH='codex/af2-luminance-safe'
REPO=Path('/content/coffee-bean-detection'); WORK=Path('/content')
os.chdir(WORK)
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96','gdown'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
import torch
if not torch.cuda.is_available(): raise RuntimeError('Aktifkan GPU Colab')
from coffee_detector.drive_project import resolve_drive_project_root
from coffee_detector.analysis.coffee_standard_j25_thesis_provenance import audit_j25_thesis_provenance
from coffee_detector.data.prepare_coffee_standard_j25_source_split import prepare_j25_source_split
PROJECT=resolve_drive_project_root()
def valid_report_root(folder, report_name, protocol):
    candidates=[PROJECT/'experiments'/folder]
    candidates += list(Path('/content/drive/.shortcut-targets-by-id').glob(f'*/Coffee_Bean_Detection/experiments/{folder}'))
    valid=[]
    for root in candidates:
        report=root/'val_reports'/report_name
        if report.is_file():
            payload=json.loads(report.read_text())
            if payload.get('protocol')==protocol and payload.get('test_images_accessed') is False: valid.append(root)
    unique={str(path):path for path in valid}
    return sorted(unique.values(),key=lambda path:(0 if str(path).startswith(str(PROJECT)) else 1,len(str(path))))[0] if unique else None
DIRECT=valid_report_root('coffee-standard-j25-af2-direct-v2','D0DIRECT_seed42_result.json','coffee-standard-j25-train-siblings-af2-direct-seed42-v2')
LUMINANCE=valid_report_root('coffee-standard-j25-af2-luminance-v1','AF2LUMDIRECT_seed42_result.json','coffee-standard-j25-af2-luminance-isolation-seed42-v1')
if DIRECT is not None: PROJECT=DIRECT.parents[1]
ARCHIVE=WORK/'data_aug_11.zip'
if not ARCHIVE.is_file(): subprocess.run([sys.executable,'-m','gdown','https://drive.google.com/uc?id=1AofT7VbiNFM8ul-0vyCAKj7Rp4j5OX0f','-O',str(ARCHIVE)],check=True)
PROVENANCE=WORK/'coffee_standard_j25_thesis_provenance.json'
provenance=audit_j25_thesis_provenance(ARCHIVE,PROVENANCE)
if not provenance['decision'].startswith('PASS'): raise RuntimeError(f'Provenance gagal: {provenance["decision"]}')
DATA=WORK/'coffee-standard-j25-train-siblings-v2'
if DATA.exists(): shutil.rmtree(DATA)
contract=prepare_j25_source_split(ARCHIVE,DATA,seed=42,retain_train_siblings=True)
CONTRACT=DATA/'coffee_standard_j25_train_siblings_summary.json'
from ultralytics import YOLO
_=YOLO('yolo26n.pt'); PRETRAINED=REPO/'yolo26n.pt'
OUT=PROJECT/'experiments/coffee-standard-j25-af2-luminance-safe-v1'; OUT.mkdir(parents=True,exist_ok=True)
print('ARM:',ARM,'| GPU:',torch.cuda.get_device_name(0),'| DATA:',contract['images'],'| OUT:',OUT)


In [ ]:
from coffee_detector.experiments.run_coffee_standard_j25_af2_luminance_safe import run_static_preflight, build_sampler_audit
STATIC=OUT/'static_preflight.json'; SAMPLER=OUT/'sampler_audit.json'
static=run_static_preflight(PRETRAINED,STATIC,seed=42)
sampler=build_sampler_audit(DATA,SAMPLER)
print('STATIC:',static['decision'],static['gates'])
print('SAMPLER:',sampler['decision'],'| images:',sampler['images'],'| identities:',sampler['source_identities'])
assert static['decision']=='PASS' and sampler['decision']=='PASS', 'STOP: preflight gagal; training tidak dijalankan.'


In [ ]:
LOG=OUT/f'{ARM}_seed42_run.log'
command=[sys.executable,'-u','-m','coffee_detector.experiments.run_coffee_standard_j25_af2_luminance_safe','--data-root',str(DATA),'--development-contract',str(CONTRACT),'--provenance-summary',str(PROVENANCE),'--pretrained-checkpoint',str(PRETRAINED),'--output-root',str(OUT),'--seed','42','--device','0','--authorize-training']
print('START/RESUME:',ARM,'| log=',LOG,flush=True)
with LOG.open('a',encoding='utf-8') as stream: process=subprocess.Popen(command,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT)
last=-1
while process.poll() is None:
    results=OUT/ARM/f'{ARM}_seed42'/'results.csv'
    epochs=sum(1 for _ in csv.DictReader(results.open(encoding='utf-8'))) if results.is_file() else 0
    if epochs!=last: print(f'{ARM}: {epochs}/50 epoch tercatat',flush=True); last=epochs
    time.sleep(120)
if process.returncode:
    print('\n'.join(LOG.read_text(errors='replace').splitlines()[-120:])); raise RuntimeError(f'{ARM} gagal: {process.returncode}')
RESULT=OUT/'val_reports'/f'{ARM}_seed42_result.json'
print(json.dumps(json.loads(RESULT.read_text()),indent=2,ensure_ascii=False))
print('last.pt tersimpan di Drive setiap epoch. Test tidak diekstrak.')


In [ ]:
from coffee_detector.experiments.run_coffee_standard_j25_af2_luminance_safe import build_decision
if DIRECT is None or LUMINANCE is None:
    print('Training selesai. Reference D0/AF2LUM belum terlihat pada akun ini; kirim RESULT di atas dan jangan ulang training.')
else:
    DECISION=OUT/'af2_luminance_safe_seed42_decision.json'
    result=build_decision(DIRECT,LUMINANCE,OUT,DECISION)
    print('VALUES:',result['values'])
    print('VS D0:',result['deltas_vs_d0'])
    print('VS AF2LUM:',result['deltas_vs_af2_luminance'])
    print('DECISION:',result['decision'],'| NEXT:',result['next'],'| TEST:',result['test_opened'])
